In [ ]:
import json
import random
import numpy as np
import torch

import torch.distributed.fsdp
class FakeFSDPModule:
    pass
torch.distributed.fsdp.FSDPModule = FakeFSDPModule

from datasets import Dataset
from transformers import Trainer, TrainingArguments
from unsloth import FastLanguageModel

Unknown instance spec: Please select VM configuration

In [ ]:
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"
OUTPUT_DIR = "biling_model_test"
SAVE_DIR = "biling_model"

MAX_SEQ_LENGTH = 1024
SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
with open("train_bilingual_shuffled.txt", "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw_data)} train examples")

Loaded 42806 train examples


In [ ]:
LABELS = sorted(list(set(x["relation"] for x in raw_data)))
relation_list_str = ", ".join(LABELS)

print("Labels:", LABELS)

Labels: ['ABBREVIATION', 'AFFECTS', 'ALTERNATIVE_NAME', 'APPLIED_TO', 'ASSOCIATED_WITH', 'FINDING_OF', 'HAS_CAUSE', 'ORIGINS_FROM', 'PART_OF', 'PHYSIOLOGY_OF', 'SUBCLASS_OF', 'TO_DETECT_OR_STUDY', 'TREATED_USING', 'USED_IN', 'no_relation']


In [ ]:
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.4.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.325 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 491.83it/s]
Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
prompt_template = """You are an expert in biomedical information extraction.
Analyze the text and determine the relation between the two specified entities.
You must choose ONLY ONE relation from the following list:
[{relation_list}]

Text: {text}
Entity 1 (Head): {head_name} (Type: {head_type})
Entity 2 (Tail): {tail_name} (Type: {tail_type})
Relation:"""

In [ ]:
def build_prompt(example):
    return prompt_template.format(
        relation_list=relation_list_str,
        text=example["text"],
        head_name=example["h"]["name"],
        head_type=example["head_type"],
        tail_name=example["t"]["name"],
        tail_type=example["tail_type"],
    )

def tokenize_completion_only(example):
    """
    Создаем:
    - input_ids = prompt + " " + label + eos
    - labels = -100 на prompt части, target токены только на answer части
    """

    prompt = build_prompt(example)
    answer = " " + example["relation"] + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )["input_ids"]

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
        truncation=True,
        max_length=64,
    )["input_ids"]

    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)

    labels = [-100] * len(prompt_ids) + answer_ids

    if len(input_ids) > MAX_SEQ_LENGTH:
        input_ids = input_ids[:MAX_SEQ_LENGTH]
        attention_mask = attention_mask[:MAX_SEQ_LENGTH]
        labels = labels[:MAX_SEQ_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "prompt_text": prompt,
        "answer_text": answer,
    }

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(tokenize_completion_only)

print("\n--- EXAMPLE PROMPT ---")
print(dataset[0]["prompt_text"])
print("\n--- EXAMPLE ANSWER ---")
print(dataset[0]["answer_text"])
print("----------------------\n")

Map: 100%|██████████| 42806/42806 [00:40<00:00, 1069.36 examples/s]


--- EXAMPLE PROMPT ---
You are an expert in biomedical information extraction.
Analyze the text and determine the relation between the two specified entities.
You must choose ONLY ONE relation from the following list:
[ABBREVIATION, AFFECTS, ALTERNATIVE_NAME, APPLIED_TO, ASSOCIATED_WITH, FINDING_OF, HAS_CAUSE, ORIGINS_FROM, PART_OF, PHYSIOLOGY_OF, SUBCLASS_OF, TO_DETECT_OR_STUDY, TREATED_USING, USED_IN, no_relation]

Text: In the presented herein clinical case report, a female patient with arteriovenous angiodysplasia of the lower limb with the tibial bone involved into the pathological process underwent repeated stagewise embolisations, failing however to achieve complete liquidation of the arteriovenous reflux.
Entity 1 (Head): bone (Type: ANATOMY)
Entity 2 (Tail): limb (Type: ANATOMY)
Relation:

--- EXAMPLE ANSWER ---
 no_relation<|end_of_text|>
----------------------



In [ ]:
class CompletionOnlyCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id
        if self.pad_token_id is None:
            self.pad_token_id = tokenizer.eos_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        input_ids_batch = []
        attention_mask_batch = []
        labels_batch = []

        for f in features:
            input_ids = f["input_ids"]
            attention_mask = f["attention_mask"]
            labels = f["labels"]

            pad_len = max_len - len(input_ids)

            input_ids = input_ids + [self.pad_token_id] * pad_len
            attention_mask = attention_mask + [0] * pad_len
            labels = labels + [-100] * pad_len

            input_ids_batch.append(input_ids)
            attention_mask_batch.append(attention_mask)
            labels_batch.append(labels)

        batch = {
            "input_ids": torch.tensor(input_ids_batch, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_batch, dtype=torch.long),
            "labels": torch.tensor(labels_batch, dtype=torch.long),
        }
        return batch

data_collator = CompletionOnlyCollator(tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    warmup_ratio=0.1,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer_stats = trainer.train()

 90%|█████████ | 1810/2007 [4:28:55<28:15,  8.61s/it]

{'loss': '0.01898', 'grad_norm': '0.2524', 'learning_rate': '1.096e-05', 'epoch': '2.706'}


 91%|█████████ | 1820/2007 [4:30:22<26:52,  8.62s/it]

{'loss': '0.02326', 'grad_norm': '0.5298', 'learning_rate': '1.041e-05', 'epoch': '2.721'}


 91%|█████████ | 1830/2007 [4:31:49<25:27,  8.63s/it]

...

{'loss': '0.03138', 'grad_norm': '0.5859', 'learning_rate': '9.967e-07', 'epoch': '2.975'}


100%|█████████▉| 2000/2007 [4:57:03<01:00,  8.63s/it]

{'loss': '0.02978', 'grad_norm': '0.4252', 'learning_rate': '4.43e-07', 'epoch': '2.99'}


100%|██████████| 2007/2007 [4:58:17<00:00,  8.92s/it]

{'train_runtime': '1.79e+04', 'train_samples_per_second': '7.175', 'train_steps_per_second': '0.112', 'train_loss': '0.08927', 'epoch': '3'}


In [ ]:
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to: {SAVE_DIR}")


Model saved to: biling_model
